# M2 Notebook Release Header
**Release Header (Auto)**
- Purpose: analysis workflow notebook.
- Inputs: local source tables and matrices.
- Outputs: figures and summary metrics.
- Dependencies: existing scientific Python packages only.
- Execution Order: run top-to-bottom.


# Publish Notebook Header

- Purpose: Reproducible analysis notebook for M2 release package.
- Inputs: Local project data files (configured via relative paths or project root variable).
- Outputs: Analysis tables and figures with unchanged logic/style.
- Execution Order: Run cells top-to-bottom.
- Privacy: Local identity/path tokens are anonymized for release.


## Section: Core Analysis
This section preserves original computation and visualization behavior.


In [19]:
import pandas as pd
import numpy as np
import os
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib.colors import ListedColormap


In [20]:
def correlation_index(spikes_i, spikes_j):
    """
    spikes_i, spikes_j: 1D binary arrays of same length
    """
    T = len(spikes_i)
    N_i = spikes_i.sum()
    N_j = spikes_j.sum()
    
    N_expected = (N_i * N_j) / T
    N_coinc = np.logical_and(spikes_i, spikes_j).sum()
    
    # Avoid division by zero
    denominator = min(N_i, N_j) - N_expected
    if denominator == 0:
        return 0.0
    
    Ci = (N_coinc - N_expected) / denominator
    return np.clip(Ci, -1, 1)

In [21]:
from scipy.spatial.distance import cdist
def sts_pair_fast(spike_times_i, spike_times_j, dt):
    if len(spike_times_i) == 0 or len(spike_times_j) == 0:
        return 0.0
    
    ti = np.array(spike_times_i).reshape(-1, 1)
    tj = np.array(spike_times_j).reshape(-1, 1)
    
    # Compute all pairwise |ti - tj|
    dists = cdist(ti, tj, metric='cityblock')  # L1 distance = |ti - tj|
    
    # Check if any distance <= dt for each spike
    matched_i = np.any(dists <= dt, axis=1)  # shape: (len(ti),)
    matched_j = np.any(dists <= dt, axis=0)  # shape: (len(tj),)
    
    count_i = matched_i.sum()
    count_j = matched_j.sum()
    
    sts = (count_i + count_j) / (len(ti) + len(tj))
    return np.clip(sts, 0, 1)

In [22]:
def compute_sts_matrix(spike_trains, dt):
    """
    spike_trains: list of lists, e.g., [train1, train2, ..., trainN]
    dt: time window (same units as spike times)
    Returns: N x N STS matrix (symmetric, diagonal = 1)
    """
    n = len(spike_trains)
    sts_mat = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            if i == j:
                sts_mat[i, j] = 1.0
            else:
                sts_val = sts_pair_fast(spike_trains[i], spike_trains[j], dt)
                sts_mat[i, j] = sts_val
                sts_mat[j, i] = sts_val
    
    return sts_mat

def mean_sts(sts_mat):
    """Return scalar: average STS over all unique pairs"""
    n = sts_mat.shape[0]
    return sts_mat[np.triu_indices(n, k=0)].mean()

In [ ]:
project_path = projectpath = '${PROJECT_ROOT}'
model = 'test2'
group = 'depressed'
mice_ids = [
    #  'LHQ30'  # line for testing
    "CSDS0126","LH0167","CSDS5776","CSDS5797","CSDSQ26","LHQ30","LHQ50","LH5798" #   exclude = CSDS0087,CSDS0179,LH5798
    ]
condition = 'rescue'  # 'pre', 'post', 'rescue'
KNN_accuracy_df = pd.DataFrame(columns=['2D Accuracy','3D Accuracy'])
time_window = 4
half_win_bins = int(time_window )
# norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
cmap = 'RdBu_r'  # Red (positive) - White (0) - Blue (negative)
all_mice_MSTM_matrices = []
record_time = 1 #1s

for miceID in mice_ids:
    if miceID == mice_ids[0]:
        VERBOSE = True
    signal_path_TST_align = os.path.join(projectpath, 'signal_data', 'test2', condition, miceID, 'signal_save', f'{miceID}_{condition}_all_cell_table_TST.csv')
    mice_out = os.path.join(projectpath, 'correlation_analysis', model,condition,miceID)
    if not os.path.exists(signal_path_TST_align):
        print(f"File not found: {signal_path_TST_align}, skipping {miceID}")
        continue

    print(f"Processing {miceID} - {condition}")
    TST_data = pd.read_csv(signal_path_TST_align)

    T = TST_data.shape[0]
    time_axis = np.arange(T) * record_time

    plt.imshow(TST_data.T,aspect='auto',cmap='coolwarm')
    plt.title(f'Spike Train Data for {miceID} - {condition}')
    plt.show()

    binary_data = (TST_data>1).astype(int)
    plt.imshow(binary_data.T,aspect='auto',cmap='gray_r')
    plt.show()
    print(f'Binarized Spike Train Data for {miceID} - {condition} (threshold Z>1)')

    spike_trains = [] # EN: translated release comment.
    for neuron in binary_data.columns:
        
        spike_series = binary_data[neuron].values  # shape: (T,)
        
        # spike
        spike_indices = np.where(spike_series == 1)[0] # EN: translated release comment.
        
        
        spike_times = time_axis[spike_indices]
        
        spike_trains.append(spike_times.tolist()) # EN: translated release comment.
    print(f"Computed spike trains for {len(spike_trains)} neurons.")
    print(f"Example spike train for neuron 0: {spike_trains[0]} ...")

    sts_matrix = compute_sts_matrix(spike_trains, dt=2)
    plt.imshow(sts_matrix, aspect='auto', cmap='viridis', vmin=0, vmax=1)
    plt.title(f'Spike Time Tiling Coefficient (STS) Matrix for {miceID} - {condition}')
    plt.colorbar(label='STS')
    plt.show()
    sts_matrix_df = pd.DataFrame(sts_matrix, index=TST_data.columns, columns=TST_data.columns)
    sts_matrix_df.to_csv(os.path.join(mice_out, f'{miceID}_{condition}_sts_matrix.csv'))

    sts_average = mean_sts(sts_matrix)
    print(f"Average STS for {miceID} - {condition}: {sts_average:.4f}")


    n_neurons = TST_data.shape[1]
    Ci_matrix = np.zeros((n_neurons, n_neurons))    
    for i in range(n_neurons):
        for j in range(i+1, n_neurons):
            ci = correlation_index(binary_data.iloc[:, i].values, binary_data.iloc[:, j].values)
            Ci_matrix[i, j] = ci
            Ci_matrix[j, i] = ci # EN: translated release comment.
    ci_df = pd.DataFrame(Ci_matrix, index=TST_data.columns, columns=TST_data.columns)
    ci_df.to_csv(os.path.join(mice_out, f'{miceID}_{condition}_correlation_index_matrix.csv'))


    plt.title(f'Correlation Index Matrix for {miceID} - {condition} Average CI: {ci_df.values[np.triu_indices(n_neurons, k=0)].mean():.4f}')
    plt.imshow(ci_df,aspect='auto',cmap='viridis')
    plt.show()


    map = sns.clustermap(sts_matrix, cmap='viridis', figsize=(10, 10))
    plt.show()
    order = map.dendrogram_row.reordered_ind
    ordered_data = TST_data.iloc[:, order]
    plt.imshow(ordered_data.T,aspect='auto',cmap='coolwarm')
    plt.title(f'Ordered Spike Train Data for {miceID} - {condition} by STS Clustering')
    plt.show()


    all_mice_MSTM_matrices.append({
        'miceID': miceID,
        'mean_ci': ci_df.values[np.triu_indices(n_neurons, k=0)].mean(),
        'mean_sts': sts_average,
        'group': group
    })
    
    plt.imshow(ordered_data.T,aspect='auto',cmap='coolwarm')
    plt.show()

all_mice_MSTM_matrices = pd.DataFrame(all_mice_MSTM_matrices)
all_mice_MSTM_matrices.to_csv(os.path.join(project_path, 'correlation_analysis', model,condition,'group',group,f'{group}_all_mice_MSTM_matrices_{condition}.csv'), index=False)
    






In [24]:
def correlation_index(spikes_i, spikes_j, tau_bins=0):
    """
    spikes_i, spikes_j: 1D binary arrays of same length
    tau_bins: half-width in bins (currently only supports tau_bins=0 for simplicity)
    """
    T = len(spikes_i)
    N_i = spikes_i.sum()
    N_j = spikes_j.sum()
    
    if tau_bins == 0:
        N_coinc = np.logical_and(spikes_i, spikes_j).sum()
    else:
        # （，）
        raise NotImplementedError("tau_bins > 0 not implemented here")
    
    N_expected = (N_i * N_j) / T
    
    # Avoid division by zero
    denominator = min(N_i, N_j) - N_expected
    if denominator == 0:
        return 0.0
    
    Ci = (N_coinc - N_expected) / denominator
    return np.clip(Ci, -1, 1)